In [ ]:
# ---- Cell 1: install tools, write the OASIS-2 table, build the OASIS-1 template ----
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'antspyx', 'antspynet'], capture_output=True, text=True)
print('pip install finished, code', r.returncode); print(r.stderr[-600:])
import os, glob, numpy as np, pandas as pd
import ants, antspynet, nibabel as nib
print('ants', ants.__version__)
 
# The OASIS-2 subject table (subject_id, session_id, age, sex, educ, educ_years, CDR, y_dementia, converted_later)
# is NOT stored in this repository because of the OASIS data use terms.
# Build it from the official OASIS-2 demographics file (see the README) and put its path here:
SUBJECTS_OASIS2_CSV = '/kaggle/input/YOUR-DATASET/subjects_oasis2.csv'
os.makedirs('/kaggle/working/o2', exist_ok=True)
import shutil; shutil.copy(SUBJECTS_OASIS2_CSV, '/kaggle/working/o2/subjects_oasis2.csv')
subj2 = pd.read_csv('/kaggle/working/o2/subjects_oasis2.csv')
print('OASIS-2 table:', subj2.shape, '| dementia =', int(subj2.y_dementia.sum()), '| controls =', int((1 - subj2.y_dementia).sum()))
 
imgs = sorted(glob.glob('/kaggle/input/**/*_MR1/RAW/mpr-1.nifti.img', recursive=True))
print('OASIS-2 raw scans found:', len(imgs)); print(imgs[:2])
assert len(imgs) == 150, 'expected 150 scans: is the oasis2-mr1 dataset added to this notebook?'
 
npz1 = sorted(glob.glob('/kaggle/input/**/OAS1_*_MR1.npz', recursive=True))
print('OASIS-1 volumes found:', len(npz1))
assert len(npz1) > 50, 'add your oasis1-prepped dataset to this notebook too'
TEMPLATE = np.mean([np.load(f)['vol'].astype(np.float32) for f in npz1[:60]], axis=0)
print('template', TEMPLATE.shape, 'brain voxels', int((TEMPLATE > -1).sum()))
np.save('/kaggle/working/o2/template.npy', TEMPLATE)

In [ ]:
# ---- Cell 2 (version 3): trial on 3 scans (skull-strip + align to OASIS-1 space), then LOOK at the pictures ----
import time, matplotlib.pyplot as plt
from scipy import ndimage

def load_ras(img_path):
    """read the raw scan and put it in a standard orientation (RAS). returns an ants image."""
    im = nib.as_closest_canonical(nib.load(img_path))
    data = np.asarray(im.dataobj).astype(np.float32)
    while data.ndim > 3: data = data[..., 0]
    tmp = '/kaggle/working/o2/tmp_in.nii.gz'
    nib.save(nib.Nifti1Image(data, im.affine), tmp)
    return ants.image_read(tmp)

def largest(mask):
    lab, n = ndimage.label(mask)
    if n <= 1: return mask
    sizes = ndimage.sum(mask, lab, range(1, n + 1))
    return lab == (1 + int(np.argmax(sizes)))

def normalise(vol):
    # the SAME rule that was used for the OASIS-1 volumes
    m = vol > 0
    assert m.sum() > 1000, 'empty brain after alignment (%d voxels)' % int(m.sum())
    v = vol[m]; lo, hi = np.percentile(v, [1, 99.5]); v = np.clip(v, lo, hi)
    mu, sd = v.mean(), v.std() + 1e-6
    out = np.zeros_like(vol, dtype=np.float32)
    out[m] = (np.clip(vol[m], lo, hi) - mu) / sd
    return out

def com(a):
    return np.argwhere(a != 0).mean(0)

# the template at 2 mm, used only to pick the axis directions
TARGET_VOX = float(np.median([(np.load(f)['vol'] != 0).sum() for f in npz1[:20]]))   # typical OASIS-1 brain size (1 mm voxels)
print('typical OASIS-1 brain size: %.2f million voxels' % (TARGET_VOX / 1e6))
T2 = ndimage.zoom(TEMPLATE, 0.5, order=1)
T2m = T2 != 0

def choose_flip(arr, sp):
    """try the 4 up/down and front/back flips (left-right is kept as it is); return the best + all scores"""
    best, scores = None, {}
    for ap in (False, True):
        for si in (False, True):
            a = arr
            if ap: a = np.flip(a, 1)
            if si: a = np.flip(a, 2)
            s = (TARGET_VOX / max(float((a != 0).sum() * np.prod(sp)), 1.0)) ** (1 / 3.0)
            z = ndimage.zoom(a, np.array(sp) / 2.0 * s, order=1)
            z = ndimage.shift(z, com(T2) - com(z), order=1)
            out = np.zeros_like(T2); sl = tuple(slice(0, min(x, y)) for x, y in zip(T2.shape, z.shape))
            out[sl] = z[sl]
            m = (out != 0) | T2m
            x, y = out[m] - out[m].mean(), T2[m] - T2[m].mean()
            ncc = float((x * y).sum() / (np.sqrt((x * x).sum() * (y * y).sum()) + 1e-9))
            scores[(False, ap, si)] = round(ncc, 3)
            if best is None or ncc > scores[best]: best = (False, ap, si)
    return best, scores

FIXED = ants.from_numpy(TEMPLATE, spacing=(1.0, 1.0, 1.0))
COM_FIXED = com(TEMPLATE) * 1.0

def as_fixed_frame(arr, sp):
    """array -> ants image with plain axes, placed so that its centre of mass sits on the template's"""
    origin = tuple(COM_FIXED - com(arr) * np.array(sp))
    return ants.from_numpy(np.ascontiguousarray(arr.astype('float32')), origin=origin, spacing=tuple(float(x) for x in sp))

def process(img_path, flip=None):
    t0 = time.time()
    img = load_ras(img_path)
    img = ants.n4_bias_field_correction(img)
    prob = antspynet.brain_extraction(img, modality='t1')
    mask = largest(prob.numpy() > 0.5).astype('float32')
    brain = img.numpy() * mask
    sp = np.array(img.spacing)
    scores = None
    if flip is None:
        flip, scores = choose_flip(brain, sp)
    for ax, f in enumerate(flip):
        if f: brain, mask = np.flip(brain, ax), np.flip(mask, ax)
    # step 1: make the brain the typical OASIS-1 size, centred on the template (this alone already gives a rough match)
    s = (TARGET_VOX / max(float((mask > 0).sum() * np.prod(sp)), 1.0)) ** (1 / 3.0)
    sp2 = sp * s
    mov, movm = as_fixed_frame(brain, sp2), as_fixed_frame(mask, sp2)
    rough = ants.resample_image_to_target(mov, FIXED, interp_type=0).numpy()
    roughm = ants.resample_image_to_target(movm, FIXED, interp_type=1).numpy() > 0.5
    # step 2: fine alignment, rigid first and then affine
    path, vol = 'prescaled-only', rough * roughm
    try:
        r1 = ants.registration(fixed=FIXED, moving=mov, type_of_transform='Rigid', verbose=False)
        r2 = ants.registration(fixed=FIXED, moving=mov, type_of_transform='Affine',
                               initial_transform=r1['fwdtransforms'][0], verbose=False)
        w = ants.apply_transforms(fixed=FIXED, moving=mov, transformlist=r2['fwdtransforms'], interpolator='linear').numpy()
        wm = ants.apply_transforms(fixed=FIXED, moving=movm, transformlist=r2['fwdtransforms'], interpolator='nearestNeighbor').numpy() > 0.5
        cand = w * wm
        if (cand > 0).sum() > 0.5 * TARGET_VOX:      # sane result -> use it
            path, vol = 'affine', cand
    except Exception as e:
        print('   registration problem:', repr(e)[:200])
    vol = np.where(vol > 0, vol, 0)
    return normalise(vol), time.time() - t0, flip, scores, path

trial = imgs[:3]
res = []
for p in trial:
    v, sec, fl, sc, path = process(p)
    res.append(v)
    print(os.path.basename(os.path.dirname(os.path.dirname(p))), 'done in %.0f s' % sec, '| brain voxels', int((v != 0).sum()),
          '| used:', path, '| chosen flip (LR,AP,SI):', fl, '| scores:', sc)

rows = [('OASIS-1 average', TEMPLATE)] + [('OASIS-2 trial %d' % (i + 1), r) for i, r in enumerate(res)]
fig, ax = plt.subplots(4, 3, figsize=(10, 13))
for r_i, (name, vol) in enumerate(rows):
    c = [s_ // 2 for s_ in vol.shape]
    for k, sl in enumerate([vol[:, :, c[2]], vol[:, c[1], :], vol[c[0], :, :]]):
        ax[r_i, k].imshow(sl.T, cmap='gray', origin='lower'); ax[r_i, k].axis('off')
        ax[r_i, k].set_title(name + ' | ' + ['axial', 'coronal', 'sagittal'][k], fontsize=9)
plt.tight_layout(); plt.savefig('/kaggle/working/o2/trial_check.png', dpi=110); plt.show()

In [ ]:
# ---- Cell 3: run ALL 150 OASIS-2 scans (can be re-run: it skips scans that are already done) ----
import traceback
OUT = '/kaggle/working/o2/prepped'; os.makedirs(OUT, exist_ok=True)
FLIP = (False, False, False)       # the trial pictures showed this direction is right (3 of 3 scans)

def process_all_steps(img_path):
    img = ants.n4_bias_field_correction(load_ras(img_path))
    prob = antspynet.brain_extraction(img, modality='t1')
    mask = largest(prob.numpy() > 0.5).astype('float32')
    brain = img.numpy() * mask
    sp = np.array(img.spacing)
    for ax, f in enumerate(FLIP):
        if f: brain, mask = np.flip(brain, ax), np.flip(mask, ax)
    s = (TARGET_VOX / max(float((mask > 0).sum() * np.prod(sp)), 1.0)) ** (1 / 3.0)   # make the brain the typical OASIS-1 size
    sp2 = sp * s
    mov, movm = as_fixed_frame(brain, sp2), as_fixed_frame(mask, sp2)                  # centre it on the template
    rough = ants.resample_image_to_target(mov, FIXED, interp_type=0).numpy()
    roughm = ants.resample_image_to_target(movm, FIXED, interp_type=1).numpy() > 0.5
    vol = np.where(rough * roughm > 0, rough * roughm, 0)
    return normalise(vol)

t0 = time.time(); failed = []
for i, p in enumerate(imgs):
    sid = os.path.basename(os.path.dirname(os.path.dirname(p)))
    f_out = f'{OUT}/{sid}.npz'
    if os.path.exists(f_out): continue
    try:
        v = process_all_steps(p)
        np.savez_compressed(f_out, vol=v.astype('float16'), vox=np.array([1, 1, 1], dtype='float32'))
    except Exception as e:
        failed.append((sid, repr(e)[:150])); print('FAILED', sid, repr(e)[:150])
    if (i + 1) % 10 == 0: print(f'{i+1}/{len(imgs)} done | {(time.time()-t0)/60:.0f} min', flush=True)

# ---- quality check over all finished volumes ----
files = sorted(glob.glob(f'{OUT}/*.npz'))
tm = TEMPLATE != 0; rows = []
for f in files:
    v = np.load(f)['vol'].astype(np.float32); m = (v != 0) | tm
    x, y = v[m] - v[m].mean(), TEMPLATE[m] - TEMPLATE[m].mean()
    rows.append((os.path.basename(f)[:-4], int((v != 0).sum()), float((x * y).sum() / np.sqrt((x * x).sum() * (y * y).sum()))))
qc = pd.DataFrame(rows, columns=['session_id', 'brain_voxels', 'ncc_to_template'])
qc.to_csv('/kaggle/working/o2/qc.csv', index=False)
print('\nfinished volumes:', len(qc), '| failed:', len(failed), failed[:5])
print(qc.ncc_to_template.describe().round(3).to_string())

pick = list(qc.sort_values('ncc_to_template').session_id[:4]) + list(qc.sample(4, random_state=0).session_id)
fig, ax = plt.subplots(8, 3, figsize=(9, 24))
for r_i, sid in enumerate(pick):
    v = np.load(f'{OUT}/{sid}.npz')['vol'].astype(np.float32); c = [s_ // 2 for s_ in v.shape]
    for k, sl in enumerate([v[:, :, c[2]], v[:, c[1], :], v[c[0], :, :]]):
        ax[r_i, k].imshow(sl.T, cmap='gray', origin='lower'); ax[r_i, k].axis('off')
        ax[r_i, k].set_title(('LOWEST-4 ' if r_i < 4 else 'random ') + sid + ' ' + ['axial', 'coronal', 'sagittal'][k], fontsize=8)
plt.tight_layout(); plt.savefig('/kaggle/working/o2/qc_pictures.png', dpi=80); plt.show()